In [0]:
# Environment configuration
catalog_name = "learning_projects"
schema_name = "retail_project"

table_prefix = f"{catalog_name}.{schema_name}"

# Source tables
fact_sales_df = spark.table(
    f"{table_prefix}.fact_sales"
)

dim_products_df = spark.table(
    f"{table_prefix}.dim_product"
)

In [0]:
# fact_sales_df = spark.table(
#     "learning_projects.retail_project.fact_sales"
# )
from pyspark.sql.functions import sum, countDistinct, count, month, year, col

gold_monthly_sales_df = (
    fact_sales_df
    .groupBy(
        year("order_purchase_timestamp").alias("sales_year"),
        month("order_purchase_timestamp").alias("sales_month")
    )
    .agg(
        sum("price").alias("product_sales"),
        sum("freight_value").alias("freight"),
        sum("line_total").alias("total_revenue"),
        countDistinct("order_id").alias("distinct_orders"),
        count("*").alias("order_line_count")
    )
    .orderBy(col("sales_year"), col("sales_month"))
)

In [0]:
gold_monthly_sales_df.show(10, truncate = False)

In [0]:
gold_monthly_sales_df.agg(
    sum("product_sales").alias("product_sales"),
    sum("freight").alias("freight"),
    sum("total_revenue").alias("total_revenue"),
    sum("order_line_count").alias("order_line_count")
).show()

In [0]:
# Validate monthly Gold aggregations

assert (
    monthly_totals["product_sales"] ==
    fact_totals["product_sales"]
), "Product sales mismatch"

assert (
    monthly_totals["freight"] ==
    fact_totals["freight"]
), "Freight mismatch"

assert (
    monthly_totals["total_revenue"] ==
    fact_totals["total_revenue"]
), "Revenue mismatch"

assert (
    monthly_totals["order_line_count"] ==
    fact_sales_df.count()
), "Order-item count mismatch"

print("PASS: All Gold Monthly Sales validations completed")

In [0]:
(
    gold_monthly_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{table_prefix}.gold_monthly_sales"
    )
)

In [0]:
# dim_products_df = spark.table(
#     "learning_projects.retail_project.dim_product"
# )
from pyspark.sql.functions import col, sum, countDistinct, count
gold_category_performance_df = (
    fact_sales_df.alias("f").join(
        dim_products_df.alias("p"), col("f.product_sk") == col("p.product_sk"), "left"
    )
    .groupBy(
        col("p.product_category_name").alias("product_category_name")
    )
    .agg(
        sum(col("f.price")).alias("product_sales"),
        sum(col("f.freight_value")).alias("freight"),
        sum(col("f.line_total")).alias("total_revenue"),
        countDistinct(col("f.order_id")).alias("distinct_orders"),
        count("*").alias("quantity_sold")
    )
    .orderBy(col("total_revenue").desc())
)

gold_category_performance_df.show(10, truncate = False)

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Source fact totals
fact_count = fact_sales_df.count()

fact_revenue = fact_sales_df.agg(
    spark_sum("line_total").alias("revenue")
).first()["revenue"]

# Gold category totals
category_totals = gold_category_performance_df.agg(
    spark_sum("total_revenue").alias("revenue"),
    spark_sum("quantity_sold").alias("item_count")
).first()

print("Fact sales rows:", fact_count)
print("Gold category item count:", category_totals["item_count"])

print("Fact revenue:", fact_revenue)
print("Gold category revenue:", category_totals["revenue"])

print(
    "Row count reconciliation:",
    fact_count == category_totals["item_count"]
)

print(
    "Revenue reconciliation:",
    fact_revenue == category_totals["revenue"]
)

In [0]:
gold_category_performance_df.agg(
    sum("product_sales").alias("product_sales"),
    sum("freight").alias("freight"),
    sum("total_revenue").alias("total_revenue"),
    sum("quantity_sold").alias("quantity_sold")
).show()

In [0]:
# Validate Gold Category Performance

assert (
    category_totals["item_count"] == fact_count
), "Category item count mismatch"

assert (
    category_totals["revenue"] == fact_revenue
), "Category revenue mismatch"

# Check for duplicate category groups
duplicate_categories = (
    gold_category_performance_df
    .groupBy("product_category_name")
    .count()
    .filter("count > 1")
    .count()
)

assert duplicate_categories == 0, "Duplicate category groups found"

print("PASS: All Gold Category Performance validations completed")

In [0]:
(
    gold_category_performance_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{table_prefix}.gold_category_performance"
    )
)

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Aggregate the Gold monthly data
monthly_totals = gold_monthly_sales_df.agg(
    spark_sum("product_sales").alias("product_sales"),
    spark_sum("freight").alias("freight"),
    spark_sum("total_revenue").alias("total_revenue"),
    spark_sum("order_line_count").alias("order_line_count")
).first()

# Aggregate the source Fact Sales data
fact_totals = fact_sales_df.agg(
    spark_sum("price").alias("product_sales"),
    spark_sum("freight_value").alias("freight"),
    spark_sum("line_total").alias("total_revenue")
).first()

print("Gold product sales:", monthly_totals["product_sales"])
print("Fact product sales:", fact_totals["product_sales"])

print("Gold freight:", monthly_totals["freight"])
print("Fact freight:", fact_totals["freight"])

print("Gold revenue:", monthly_totals["total_revenue"])
print("Fact revenue:", fact_totals["total_revenue"])

print("Gold order-item count:", monthly_totals["order_line_count"])
print("Fact order-item count:", fact_sales_df.count())

print(
    "Revenue reconciliation:",
    monthly_totals["total_revenue"] == fact_totals["total_revenue"]
)

In [0]:
monthly_saved_df = spark.table(
    f"{table_prefix}.gold_monthly_sales"
)

category_saved_df = spark.table(
    f"{table_prefix}.gold_category_performance"
)

print("Monthly Gold rows:", monthly_saved_df.count())
print("Category Gold rows:", category_saved_df.count())

from pyspark.sql.functions import sum as spark_sum

monthly_saved_df.agg(
    spark_sum("total_revenue").alias("monthly_revenue")
).show()

category_saved_df.agg(
    spark_sum("total_revenue").alias("category_revenue")
).show()